In [1]:
path = '/Users/philipp/Downloads/topology_feature_ssl — consolidated results - Sheet3.csv'

In [2]:
import pandas as pd

In [3]:
df = pd.read_csv(path)

In [4]:
df

,arm,task,metric,dataset,item,value
0,B0,regression,spearman,midterm,followers,-0.064
1,B0,regression,spearman,midterm,friends,-0.132
2,B0,regression,spearman,midterm,favourites,-0.047
3,B0,regression,spearman,midterm,statuses,-0.033
4,B0,regression,spearman,midterm,listed,-0.102
...,...,...,...,...,...,...
237,raw_degree,regression,spearman,covid,statuses,0.138
238,raw_degree,regression,spearman,covid,acct_age,0.001
239,raw_degree,regression,spearman,twibot20,followers,0.396
240,raw_degree,regression,spearman,twibot20,statuses,0.279


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

plt.rcParams.update({"font.size": 10, "axes.titlesize": 10.5,
                     "figure.facecolor": "white", "axes.edgecolor": "#bbb"})

In [ ]:
# Read every result CSV — this experiment's local data/ plus the sibling per-task
# dirs — into ONE deliberately-messy long frame. Columns are the union of all
# schemas (NaN where a file lacks a column); source_file / source_dir keep
# provenance. This carries every metric the consolidated sheet dropped:
# spearman, mse, rmse, mae, r2, roc_auc, accuracy, f1, plus the 2x2/probe/budget
# columns. Clean / filter it downstream as you like.
import pathlib

RESULT_DIRS = [
    "data",                            # probes, 2x2 ablation, budget sweep, floors
    "../node_regression/data",         # regression: spearman, mse, rmse, mae, r2
    "../node_classification/data",     # classification: roc_auc, accuracy, f1
    "../static_link_prediction/data",  # static link prediction: roc_auc, accuracy, f1
]
frames = []
for d in RESULT_DIRS:
    for p in sorted(pathlib.Path(d).glob("*.csv")):
        t = pd.read_csv(p)
        t["source_file"] = p.name
        t["source_dir"] = d
        frames.append(t)
raw = pd.concat(frames, ignore_index=True)
print(f"{len(frames)} files -> raw {raw.shape}")
print("columns:", list(raw.columns))
raw

In [ ]:
# Summary figure over the CLEAN consolidated table (df from cell 2).
# `raw` (cell 5) has the full metric set once you've cleaned it — swap it in here later.
ARM_ORDER = ["B0", "B1", "E1", "E2", "E2b", "E4", "E4r"]
FLOORS    = ["raw_feat", "raw_degree"]
LEVER_C   = {"B0": "#6aa02e", "B1": "#6aa02e",
             "E1": "#3a8bd6", "E2": "#3a8bd6", "E2b": "#3a8bd6",
             "E4": "#e0972b", "E4r": "#e0972b",
             "raw_feat": "#9a988f", "raw_degree": "#9a988f"}
REG_ITEMS   = ["followers", "friends", "statuses", "favourites", "listed", "acct_age"]
PROBE_ITEMS = ["count_threshold", "in_degree", "out_degree", "existence", "conjunction"]
task = lambda t: df[df.task.eq(t)]

def heat(ax, M, vcenter, vspan, fmt="{:.2f}"):
    """Annotated heatmap on a diverging (RdYlGn) scale centred at vcenter."""
    im = ax.imshow(M.values, cmap="RdYlGn", aspect="auto",
                   norm=TwoSlopeNorm(vcenter=vcenter, vmin=vcenter - vspan, vmax=vcenter + vspan))
    ax.set_xticks(range(M.shape[1])); ax.set_xticklabels(M.columns, rotation=35, ha="right")
    ax.set_yticks(range(M.shape[0])); ax.set_yticklabels(M.index)
    for (i, j), v in np.ndenumerate(M.values):
        if not np.isnan(v):
            ax.text(j, i, fmt.format(v).replace("-0.00", "0.00"), ha="center", va="center",
                    fontsize=8, color="white" if abs(v - vcenter) > vspan * 0.62 else "#222")
    ax.tick_params(length=0)
    return im

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle("topology_feature_ssl — frozen-encoder results (matched 40k, single seed)",
             fontsize=15, fontweight="bold")

# A · Regression — arm × target heatmap (mean Spearman over datasets); floors below the line
ax = axes[0, 0]
reg = (task("regression").pivot_table("value", "arm", "item", aggfunc="mean")
       .reindex(ARM_ORDER + FLOORS, axis=0).reindex(REG_ITEMS, axis=1))
im = heat(ax, reg, vcenter=0.0, vspan=0.30)
ax.axhline(len(ARM_ORDER) - 0.5, color="#333", lw=1.4)
for lbl in ax.get_yticklabels():
    if lbl.get_text() in FLOORS:
        lbl.set_color("#999"); lbl.set_style("italic")
ax.set_title("Regression · Spearman (mean over 4 datasets)\n"
             "green = positive · below line = free baselines (no encoder)", loc="left")
fig.colorbar(im, ax=ax, fraction=0.045, pad=0.03)

# B · Capability probes — arm × probe heatmap (ROC-AUC, chance 0.5)
ax = axes[0, 1]
prb = (task("probe").pivot_table("value", "arm", "item")
       .reindex(ARM_ORDER, axis=0).reindex(PROBE_ITEMS, axis=1))
im = heat(ax, prb, vcenter=0.5, vspan=0.22)
ax.set_title("Capability probes · ROC-AUC (chance = 0.5)\ngreen = above chance", loc="left")
fig.colorbar(im, ax=ax, fraction=0.045, pad=0.03)

# C · Node classification — grouped bars per dataset, with the raw-feature floor
ax = axes[1, 0]
cls = task("classification").pivot_table("value", "arm", "dataset").reindex(ARM_ORDER + ["raw_feat"])
ds, x, w = list(cls.columns), np.arange(len(cls)), 0.8 / len(cls.columns)
for k, d in enumerate(ds):
    ax.bar(x + (k - (len(ds) - 1) / 2) * w, cls[d].values, w, label=d,
           color=["#3a8bd6", "#e0972b"][k % 2], edgecolor="white", lw=0.5)
ax.axhline(0.5, ls="--", color="#888", lw=1)
ax.text(len(cls) - 0.5, 0.51, "chance", fontsize=8, color="#888", ha="right")
ax.set_xticks(x); ax.set_xticklabels(cls.index); ax.set_ylim(0, 1.02); ax.set_ylabel("ROC-AUC")
ax.set_title("Node classification · ROC-AUC", loc="left"); ax.legend(fontsize=8, frameon=False)
ax.tick_params(length=0)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)

# D · Static link prediction — bars, mean ± std over datasets, coloured by lever
ax = axes[1, 1]
slp = task("static_lp").groupby("arm").value.agg(["mean", "std"]).reindex(ARM_ORDER)
ax.bar(range(len(slp)), slp["mean"], yerr=slp["std"], capsize=3,
       color=[LEVER_C[a] for a in slp.index], edgecolor="white", lw=0.5)
ax.axhline(0.5, ls="--", color="#888", lw=1)
ax.text(len(slp) - 0.5, 0.51, "chance", fontsize=8, color="#888", ha="right")
for i, m in enumerate(slp["mean"]):
    ax.text(i, m + slp["std"].fillna(0).iloc[i] + 0.015, f"{m:.2f}", ha="center", fontsize=8)
ax.set_xticks(range(len(slp))); ax.set_xticklabels(slp.index); ax.set_ylim(0, 0.92); ax.set_ylabel("ROC-AUC")
ax.set_title("Static link prediction · ROC-AUC (mean ± std over 4 datasets)", loc="left")
ax.tick_params(length=0)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)

fig.tight_layout()
fig.savefig("results_overview.png", dpi=150, bbox_inches="tight")
plt.show()